In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install mxnet
!pip install gluonnlp==0.9.1 pandas tqdm
!pip install sentencepiece
!pip install transformers
!pip install torch
!pip install kss
!pip install 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: graphviz
    Found existing installation: graphviz 0.20.1
    Uninstalling graphviz-0.20.1:
      Successfully uninstalled graphviz-0.20.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gluonnlp: filename=gluonnlp-0.9.1-cp310-cp310-linux_x86_64.whl size=564495 sha256=e4fce5259a9c688b99e6dc3f9ce87cf61805cbd69e0daa20663f92d9a38cd9bc
  Stored in directory: /root/.cache/pip/wheels/fc/5b/9c/3295bb07f7c5544a96303a48988707816f44a536e8e1413922
Successfully built gluonnlp
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 14.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, AdamW, get_cosine_schedule_with_warmup
import pandas as pd
from sklearn.model_selection import train_test_split
from kobert_tokenizer import KoBERTTokenizer
from torch.nn.utils.rnn import pad_sequence

# Define the BERTDataset class
class BERTDataset(Dataset):
    def __init__(self, dataset, sent_idx, label_idx, max_len, tokenizer):
        self.sentences = [tokenizer.encode(i[sent_idx], max_length=max_len,
                                           truncation=True) for i in dataset]
        self.labels = [i[label_idx] for i in dataset]
        self.max_len = max_len

    def __getitem__(self, i):
        # Right-pad the token_ids up to self.max_len
        token_ids = self.sentences[i] + [tokenizer.pad_token_id] * (self.max_len - len(self.sentences[i]))
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(self.labels[i], dtype=torch.long)

    def __len__(self):
        return len(self.labels)


# Define the BERTClassifier class
class BERTClassifier(nn.Module):
    def __init__(self, bert, hidden_size=768, num_classes=27, dr_rate=None):
        super(BERTClassifier, self).__init__()
        self.bert = bert
        self.classifier = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(p=dr_rate) if dr_rate else None

    def forward(self, token_ids):
        # Create an attention mask
        # Attention mask values are 1 for non-padding tokens and 0 for padding
        attention_mask = (token_ids != tokenizer.pad_token_id).float()

        # Pass the attention mask to the BERT model
        output = self.bert(input_ids=token_ids, attention_mask=attention_mask)
        pooled_output = output.pooler_output
        if self.dropout:
            pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)



# Load and preprocess dataset
from google.colab import drive
drive.mount('/content/drive')
train_data = pd.read_csv('/content/drive/MyDrive/27_emotion_sentence.csv', encoding='utf-8')




# Label mapping. Note: Must start from "0".
label_mapping = {
    "분노": 0, "낙관": 1, "두려움": 2, "당혹감": 3, "슬픔": 4,
    "자랑스러움": 5, "혼란": 6, "반감": 7, "걱정": 8, "비통": 9,
    "안도": 10, "기쁨": 11, "실망": 12, "갈망": 13, "회한": 14,
    "혐오": 15, "흥분": 16, "놀라움": 17, "짜증": 18, "호기심": 19,
    "감사": 20, "재미": 21, "감탄": 22, "인정": 23, "배려": 24,
    "사랑": 25, "깨달음": 26
}

# Apply the mapping to labels
train_data['Emotion_2'] = train_data['Emotion_2'].map(label_mapping)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Initialize tokenizer and model
tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')
bert_model = BertModel.from_pretrained('skt/kobert-base-v1')

# Prepare datasets
max_len = 512
train_set, test_set = train_test_split(train_data[['Sentence', 'Emotion_2']].values.tolist(), test_size=0.2, random_state=4)

train_dataset = BERTDataset(train_set, 0, 1, max_len, tokenizer)
test_dataset = BERTDataset(test_set, 0, 1, max_len, tokenizer)

# Prepare data loaders
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

# Define calc_accuracy function
def calc_accuracy(out, labels):
    predictions = torch.argmax(out, dim=1)
    correct = (predictions == labels).float()
    accuracy = correct.sum() / len(correct)
    return accuracy

# Set up the optimizer and scheduler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTClassifier(bert_model, dr_rate=0.5).to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = nn.CrossEntropyLoss()

tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/369M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
from transformers import get_cosine_schedule_with_warmup

# Configuration
warmup_ratio = 0.1
num_epochs = 5
max_grad_norm = 1.0

# Create the learning rate scheduler
num_training_steps = len(train_dataloader) * num_epochs
num_warmup_steps = int(num_training_steps * warmup_ratio)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps)

In [ ]:
from tqdm import tqdm
torch.backends.cudnn.deterministic = True


# Training loop
for epoch in range(num_epochs):
    model.train()  # Tell the model to set itself to training mode
    train_acc = 0.0
    train_loss = 0.0

    for batch_id, (token_ids, labels) in enumerate(tqdm(train_dataloader, desc='Training')):
        token_ids = token_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        output = model(token_ids)  # Pass token_ids and the model will handle the attention mask


        loss = loss_fn(output, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        scheduler.step()  # Update learning rate schedule

        train_acc += calc_accuracy(output, labels)
        train_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss/(batch_id+1)} | Train Accuracy: {train_acc/(batch_id+1)}")

    # Validation phase
    model.eval()
    test_acc = 0.0

    with torch.no_grad():
        for batch_id, (token_ids, labels) in enumerate(tqdm(test_dataloader, desc='Validation')):
            token_ids = token_ids.to(device)
            labels = labels.to(device)

            output = model(token_ids)  # Pass only token_ids
            test_acc += calc_accuracy(output, labels)

    print(f"Epoch {epoch+1}/{num_epochs} | Test Accuracy: {test_acc/(batch_id+1)}")


Training: 100%|██████████| 1487/1487 [35:43<00:00,  1.44s/it]


Epoch 1/5 | Train Loss: 1.9960091687643697 | Train Accuracy: 0.4090750515460968


Validation: 100%|██████████| 372/372 [03:07<00:00,  1.98it/s]


Epoch 1/5 | Test Accuracy: 0.5815972089767456


Training: 100%|██████████| 1487/1487 [35:38<00:00,  1.44s/it]


...

Epoch 4/5 | Test Accuracy: 0.6331204771995544


Training: 100%|██████████| 1487/1487 [35:37<00:00,  1.44s/it]


Epoch 5/5 | Train Loss: 0.40655349591332307 | Train Accuracy: 0.8913320302963257


Validation: 100%|██████████| 372/372 [03:08<00:00,  1.98it/s]

Epoch 5/5 | Test Accuracy: 0.6255600452423096


In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Validation phase
model.eval()
test_acc = 0.0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_id, (token_ids, labels) in enumerate(tqdm(test_dataloader, desc='Validation')):
        token_ids = token_ids.to(device)
        labels = labels.to(device)

        output = model(token_ids)  # Pass only token_ids
        test_acc += calc_accuracy(output, labels)

        # Store predictions and true labels
        preds = torch.argmax(output, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"Epoch {epoch+1}/{num_epochs} | Test Accuracy: {test_acc/(batch_id+1)}")

# Calculate and print the classification report
print(classification_report(all_labels, all_preds))

Validation: 100%|██████████| 372/372 [03:28<00:00,  1.78it/s]

Epoch 5/5 | Test Accuracy: 0.6255600452423096
              precision    recall  f1-score   support

           0       0.71      0.69      0.70      1125
           1       0.86      0.35      0.50        17
           2       0.57      0.53      0.55       116
           3       0.33      0.10      0.15        50
           4       0.64      0.69      0.67       840
           5       0.77      0.42      0.54        24
           6       0.50      0.52      0.51       444
           7       0.30      0.32      0.31       481
           8       0.68      0.69      0.68       320
           9       0.00      0.00      0.00         9
          10       0.88      0.58      0.70        12
          11       0.71      0.66      0.68       511
          12       0.00      0.00      0.00         8
          13       0.50      0.17      0.25        12
          14       0.33      0.20      0.25         5
          15       0.58      0.56      0.57       252
          16       0.68      0.60  


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# Save the model if needed
torch.save(model.state_dict(), '/content/drive/MyDrive/emotion_class_model.pth')

In [2]:
#신규 데이터 학습을 위한 튜닝모델 불러오기(라이브러리 설치)

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, AdamW, get_cosine_schedule_with_warmup
import pandas as pd
from sklearn.model_selection import train_test_split
from kobert_tokenizer import KoBERTTokenizer
from torch.nn.utils.rnn import pad_sequence

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
#저장한 모델 불러오기
device = "cuda" if torch.cuda.is_available() else "cpu"
model =  BertModel.from_pretrained('skt/kobert-base-v1')
model_state_dict = torch.load("/content/drive/MyDrive/emotion_class_model.pth", map_location=device)
model.eval()

config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/369M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(8002, 768, padding_idx=1)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
   

In [5]:
#새로운 데이터 불러오기
new_data = pd.read_csv('/content/drive/MyDrive/감성대화말뭉치_Training.csv', encoding='utf-8')

In [6]:
# Define the BERTDataset class
class BERTDataset(Dataset):
    def __init__(self, dataset, sent_idx, label_idx, max_len, tokenizer):
        self.sentences = [tokenizer.encode(i[sent_idx], max_length=max_len,
                                           truncation=True) for i in dataset]
        self.labels = [i[label_idx] for i in dataset]
        self.max_len = max_len

    def __getitem__(self, i):
        # Right-pad the token_ids up to self.max_len
        token_ids = self.sentences[i] + [tokenizer.pad_token_id] * (self.max_len - len(self.sentences[i]))
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(self.labels[i], dtype=torch.long)

    def __len__(self):
        return len(self.labels)


# Define the BERTClassifier class
class BERTClassifier(nn.Module):
    def __init__(self, bert, hidden_size=768, num_classes=27, dr_rate=None):
        super(BERTClassifier, self).__init__()
        self.bert = bert
        self.classifier = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(p=dr_rate) if dr_rate else None

    def forward(self, token_ids):
        # Create an attention mask
        # Attention mask values are 1 for non-padding tokens and 0 for padding
        attention_mask = (token_ids != tokenizer.pad_token_id).float()

        # Pass the attention mask to the BERT model
        output = self.bert(input_ids=token_ids, attention_mask=attention_mask)
        pooled_output = output.pooler_output
        if self.dropout:
            pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)


In [7]:
new_data['2차분류'].unique()

array(['분노', '낙관', '두려움', '당혹감', '슬픔', '자랑스러움', '혼란', '반감', '걱정', '비통',
       '안도', '기쁨', '실망', '갈망', '회한', '혐오', '흥분', '놀라움', '짜증', '호기심', '감사',
       '재미', '감탄', '인정', '배려', '사랑', '깨달음'], dtype=object)

In [8]:
# Initialize tokenizer and model
tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')
bert_model = BertModel.from_pretrained('skt/kobert-base-v1')

# Prepare datasets
new_data.loc[(new_data['2차분류'] == "분노"), '2차분류'] = 0  # anger → 1
new_data.loc[(new_data['2차분류'] == "낙관"), '2차분류'] = 1  # optimism → 2
new_data.loc[(new_data['2차분류'] == "두려움"), '2차분류'] = 2  # fear → 3
new_data.loc[(new_data['2차분류'] == "당혹감"), '2차분류'] = 3  # embarrassment → 4
new_data.loc[(new_data['2차분류'] == "슬픔"), '2차분류'] = 4  # sadness → 5
new_data.loc[(new_data['2차분류'] == "자랑스러움"), '2차분류'] = 5  # pride → 6
new_data.loc[(new_data['2차분류'] == "혼란"), '2차분류'] = 6  # confusion → 7
new_data.loc[(new_data['2차분류'] == "반감"), '2차분류'] = 7  # disapproval → 8
new_data.loc[(new_data['2차분류'] == "걱정"), '2차분류'] = 8  # nervousness → 9
new_data.loc[(new_data['2차분류'] == "비통"), '2차분류'] = 9  # grief → 10
new_data.loc[(new_data['2차분류'] == "안도"), '2차분류'] = 10  # relif → 11
new_data.loc[(new_data['2차분류'] == "기쁨"), '2차분류'] = 11  # joy → 12
new_data.loc[(new_data['2차분류'] == "실망"), '2차분류'] = 12  # disappointment → 13
new_data.loc[(new_data['2차분류'] == "갈망"), '2차분류'] = 13  # desire → 14
new_data.loc[(new_data['2차분류'] == "회한"), '2차분류'] = 14  # remorse → 15
new_data.loc[(new_data['2차분류'] == "혐오"), '2차분류'] = 15  # disgust → 16
new_data.loc[(new_data['2차분류'] == "흥분"), '2차분류'] = 16  # excitement → 17
new_data.loc[(new_data['2차분류'] == "놀라움"), '2차분류'] = 17  # surprise → 18
new_data.loc[(new_data['2차분류'] == "짜증"), '2차분류'] = 18  # annoyance → 19
new_data.loc[(new_data['2차분류'] == "호기심"), '2차분류'] = 19  # curiosity → 20
new_data.loc[(new_data['2차분류'] == "감사"), '2차분류'] = 20  # gratitude → 21
new_data.loc[(new_data['2차분류'] == "재미"), '2차분류'] = 21  # amusement → 22
new_data.loc[(new_data['2차분류'] == "감탄"), '2차분류'] = 22  # admiration → 23
new_data.loc[(new_data['2차분류'] == "인정"), '2차분류'] = 23  # approval → 24
new_data.loc[(new_data['2차분류'] == "배려"), '2차분류'] = 24  # caring → 25
new_data.loc[(new_data['2차분류'] == "사랑"), '2차분류'] = 25  # love → 26
new_data.loc[(new_data['2차분류'] == "깨달음"), '2차분류'] = 26  # realization → 27

# Prepare datasets
data_list = []
for q, label in zip(new_data['F_Sentence'], new_data['2차분류']) :
    data = []
    data.append(q)
    data.append(str(label))

    data_list.append(data)

split_sentences = []
corpus = new_data['F_Sentence']


tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


In [9]:
test_df = pd.DataFrame(data_list)
test_df.head()

,0,1
0,일은 왜 해도 해도 끝이 없을까? 화가 난다.많이 힘드시겠어요. 주위에 의논할 상대...,0
1,이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나....,0
2,회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...,0
3,직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 ...,0
4,얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나.무시하는 것 같은...,0


In [10]:
test_df.rename(columns = {0:"text", 1:"label"}, inplace=True)

In [11]:
test_df.head()

,text,label
0,일은 왜 해도 해도 끝이 없을까? 화가 난다.많이 힘드시겠어요. 주위에 의논할 상대...,0
1,이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나....,0
2,회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...,0
3,직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 ...,0
4,얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나.무시하는 것 같은...,0


In [12]:
from transformers import TextClassificationPipeline

# Load Fine-tuning model
loaded_tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')
loaded_model = BertModel.from_pretrained('skt/kobert-base-v1')

text_classifier = TextClassificationPipeline(
    tokenizer=loaded_tokenizer,
    model=loaded_model,
    framework='pt',
    return_all_scores=True
)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.
/usr/local/lib/python3.10/dist-packages/transformers/pipelines/text_classification.py:105: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
The model 'BertModel' is not supported for . Supported models are ['AlbertForSequenceClassification', 'BartForSequenceClassification', 'BertForSequenceClassification', 'BigBirdForSequenceClassification', 'BigBirdPegasusForSequenceClassification', 'BioGptForSequenceClassification', 'BloomForSequenceClassification', 'CamembertForSequenceClassification', 'CanineForSequenceClassification', 'LlamaForSequenceClassif

In [13]:
predicted_label_list = []
predicted_score_list = []

for text in test_df['text']:
    # predict
    preds_list = text_classifier(text)[0]

    sorted_preds_list = sorted(preds_list, key=lambda x: x['score'], reverse=True)
    predicted_label_list.append(sorted_preds_list[0]) # label
    predicted_score_list.append(sorted_preds_list[1]) # score

KeyError: ignored